# SentinelQ - Image Model Training (Google Colab)

Trains the ResNet18 defect classifier on **MVTec AD** using a GPU and exports weights
for the SentinelQ backend.

## How to run
1. Runtime -> Change runtime type -> **GPU**.
2. Runtime -> **Run all**.
3. Upload your `kaggle.json` when prompted (Kaggle -> Account -> *Create New API Token*).
4. At the end, **image_v1.zip** downloads.
5. Unzip it into the repo's `models/` folder so you have `models/image/v1/weights.pt`.
6. From `backend/`, run: `python -m scripts.register_weights --model-type image --version v1 --activate`

The model architecture here is identical to `backend/ml/image/model.py`, so the exported
`weights.pt` loads directly into the serving code.


In [ ]:
!pip -q install kagglehub scikit-learn

In [ ]:
import os
from google.colab import files

print('Upload your kaggle.json (Kaggle > Account > Create New API Token)')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
for name in list(uploaded):
    if name.endswith('.json'):
        os.replace(name, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json installed')

In [ ]:
import os
import kagglehub

# If this slug is unavailable, replace it with any Kaggle MVTec AD dataset slug.
KAGGLE_DATASET = 'ipythonx/mvtec-ad'
CATEGORY = 'bottle'

path = kagglehub.dataset_download(KAGGLE_DATASET)
print('downloaded to', path)

category_dir = None
for root, dirs, _ in os.walk(path):
    if os.path.basename(root).lower() == CATEGORY and os.path.isdir(os.path.join(root, 'train', 'good')):
        category_dir = root
        break
assert category_dir, f'Could not find {CATEGORY} with train/good under {path}'
print('category dir:', category_dir)

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import ResNet18_Weights, resnet18

CLASS_NAMES = ['good', 'defect']
DEFECT_INDEX = 1


class DefectCNN(nn.Module):
    def __init__(self, num_classes=2, dropout=0.5, pretrained=True):
        super().__init__()
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = resnet18(weights=weights)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, num_classes))

    def forward(self, x):
        return self.backbone(x)

    def freeze_backbone(self, unfreeze_last_block=True):
        for p in self.backbone.parameters():
            p.requires_grad = False
        for p in self.backbone.fc.parameters():
            p.requires_grad = True
        if unfreeze_last_block:
            for p in self.backbone.layer4.parameters():
                p.requires_grad = True

In [ ]:
import glob
import random

import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED, IMG, RS = 42, 224, 256
BATCH = 32
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

paths, labels = [], []
for p in glob.glob(os.path.join(category_dir, '**', '*.*'), recursive=True):
    if os.path.splitext(p)[1].lower() not in ('.png', '.jpg', '.jpeg', '.bmp'):
        continue
    if 'ground_truth' in p.lower():
        continue
    labels.append(0 if os.path.basename(os.path.dirname(p)).lower() == 'good' else 1)
    paths.append(p)
print('images', len(paths), '| good', labels.count(0), '| defect', labels.count(1))

ids = list(range(len(paths)))
tr, tmp = train_test_split(ids, test_size=0.4, stratify=labels, random_state=SEED)
val, te = train_test_split(tmp, test_size=0.5, stratify=[labels[i] for i in tmp], random_state=SEED)


def make_tf(train):
    if train:
        return transforms.Compose([
            transforms.Resize((RS, RS)), transforms.RandomCrop(IMG),
            transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
            transforms.RandomRotation(20), transforms.ColorJitter(0.2, 0.2, 0.1),
            transforms.ToTensor(), transforms.Normalize(MEAN, STD),
        ])
    return transforms.Compose([
        transforms.Resize((RS, RS)), transforms.CenterCrop(IMG),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD),
    ])


class DS(Dataset):
    def __init__(self, subset, train):
        self.subset, self.t = subset, make_tf(train)

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, k):
        i = self.subset[k]
        return self.t(Image.open(paths[i]).convert('RGB')), labels[i]


tl = DataLoader(DS(tr, True), batch_size=BATCH, shuffle=True, num_workers=2)
vl = DataLoader(DS(val, False), batch_size=BATCH, num_workers=2)
tel = DataLoader(DS(te, False), batch_size=BATCH, num_workers=2)

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', dev)

model = DefectCNN(dropout=0.5, pretrained=True).to(dev)
model.freeze_backbone(True)

counts = np.bincount([labels[i] for i in tr], minlength=2).astype(float)
counts[counts == 0] = 1.0
weight = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32).to(dev)
criterion = nn.CrossEntropyLoss(weight=weight)
optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=3e-4, weight_decay=1e-4)


@torch.no_grad()
def evaluate(loader):
    model.eval()
    probs, ys = [], []
    for x, y in loader:
        p = torch.softmax(model(x.to(dev)), 1)[:, DEFECT_INDEX]
        probs += p.cpu().tolist(); ys += list(np.array(y))
    probs, ys = np.array(probs), np.array(ys)
    auroc = roc_auc_score(ys, probs) if len(set(ys.tolist())) > 1 else float('nan')
    return auroc, accuracy_score(ys, (probs >= 0.5).astype(int))


EPOCHS, patience = 25, 5
best, best_state, bad = -1.0, None, 0
for epoch in range(1, EPOCHS + 1):
    model.train(); total = 0.0
    for x, y in tl:
        x, y = x.to(dev), y.to(dev)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
        total += loss.item() * x.size(0)
    au, ac = evaluate(vl)
    print(f'epoch {epoch:2d} | loss {total/len(tr):.4f} | val_auroc {au:.4f} | val_acc {ac:.4f}')
    score = au if not np.isnan(au) else ac
    if score > best:
        best, bad = score, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
        if bad >= patience:
            print('early stopping'); break

model.load_state_dict(best_state)
test_au, test_ac = evaluate(tel)
print('TEST | auroc', round(float(test_au), 4), '| acc', round(float(test_ac), 4))

In [ ]:
import json
import shutil

os.makedirs('image/v1', exist_ok=True)
torch.save(model.state_dict(), 'image/v1/weights.pt')

json.dump({
    'model_type': 'image', 'version': 'v1', 'category': CATEGORY, 'seed': SEED,
    'dropout': 0.5, 'unfreeze_last_block': True, 'batch_size': BATCH,
    'epochs': EPOCHS, 'lr': 3e-4, 'weight_decay': 1e-4, 'num_workers': 0,
}, open('image/v1/config.json', 'w'), indent=2)

json.dump({
    'val_best_score': float(best),
    'test': {'auroc': float(test_au), 'accuracy': float(test_ac)},
    'num_train': len(tr), 'source': KAGGLE_DATASET, 'category': CATEGORY,
}, open('image/v1/metrics.json', 'w'), indent=2)

shutil.make_archive('image_v1', 'zip', 'image')
from google.colab import files
files.download('image_v1.zip')
print('Done. Unzip image_v1.zip into the repo models/ folder -> models/image/v1/*')